# Lab 3: EDA on Wine Quality (Red) Dataset
 
## Complete Dataset Description
- **Dataset Name:** Wine Quality (Red)
- **Source:** UCI Machine Learning Repository
- **Samples:** 1599 rows (each row represents a wine sample)
- **Features:** 11 chemical properties + 1 quality score
- **Variables:**
    - fixed acidity
    - volatile acidity
    - citric acid
    - residual sugar
    - chlorides
    - free sulfur dioxide
    - total sulfur dioxide
    - density
    - pH
    - sulphates
    - alcohol
    - quality (score from 0 to 10)
- **Description:**
This dataset contains physicochemical properties and quality ratings for red wine samples from Portugal. Each sample is evaluated by experts and assigned a quality score. The goal is to analyze the relationships between chemical features and wine quality, detect patterns, and visualize distributions. It is widely used for regression, classification, and EDA tasks in data science.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print('✓ Libraries imported successfully!')

In [2]:
# Load the dataset (UCI format uses semicolon separator)
df = pd.read_csv('winequality-red.csv', sep=';')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

,"fixed acidity;""volatile acidity"";""citric acid"";""residual sugar"";""chlorides"";""free sulfur dioxide"";""total sulfur dioxide"";""density"";""pH"";""sulphates"";""alcohol"";""quality"""
0,7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5
1,7.8;0.88;0;2.6;0.098;25;67;0.9968;3.2;0.68;9.8;5
2,7.8;0.76;0.04;2.3;0.092;15;54;0.997;3.26;0.65;...
3,11.2;0.28;0.56;1.9;0.075;17;60;0.998;3.16;0.58...
4,7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5


In [ ]:
# Check data structure
print('Dataset Information:')
print('='*80)
df.info()
print('\nStatistical Summary:')
print('='*80)
df.describe().T

In [ ]:
# Handle missing values
print('Missing Values Analysis:')
print('='*80)
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

# Visualize missing values
if missing.sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
    plt.title('Missing Values Heatmap')
    plt.show()
    df = df.dropna()
    print(f'\n✓ Dropped rows with missing values. New shape: {df.shape}')
else:
    print('\n✓ No missing values found!')

In [ ]:
# Remove duplicates
print('Duplicate Rows Analysis:')
print('='*80)
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')
print(f'Percentage: {(duplicates/len(df)*100):.2f}%')

if duplicates > 0:
    df_before = df.shape[0]
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'\n✓ Removed {df_before - df.shape[0]} duplicates')
    print(f'✓ New shape: {df.shape}')
else:
    print('\n✓ No duplicates found!')

## Additional Data Structure Analysis

In [ ]:
# Display column names and data types
print('Column Names and Data Types:')
print('='*80)
for i, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    print(f'{i:2d}. {col:25s} - {dtype}')

print(f'\nDataset Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB')

In [ ]:
# Quality distribution statistics
print('Quality Score Distribution:')
print('='*80)
quality_dist = df['quality'].value_counts().sort_index()
print(quality_dist)
print(f'\nMean Quality: {df["quality"].mean():.2f}')
print(f'Median Quality: {df["quality"].median():.0f}')
print(f'Mode Quality: {df["quality"].mode()[0]}')
print(f'Quality Range: {df["quality"].min()} - {df["quality"].max()}')

## Data Visualizations (Seaborn & Matplotlib)

In [ ]:
# Quality distribution (Seaborn)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
quality_counts = df['quality'].value_counts().sort_index()
axes[0].bar(quality_counts.index, quality_counts.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Quality Score', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Quality Distribution (Bar Chart)', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Count plot with Seaborn
sns.countplot(x='quality', data=df, palette='viridis', ax=axes[1], edgecolor='black')
axes[1].set_xlabel('Quality Score', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Quality Distribution (Seaborn)', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Alcohol vs Quality (Matplotlib & Seaborn)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot with Seaborn
sns.boxplot(x='quality', y='alcohol', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Alcohol Content by Quality (Seaborn Boxplot)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Quality Score', fontsize=12)
axes[0].set_ylabel('Alcohol (%)', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)

# Regression plot
sns.regplot(x='alcohol', y='quality', data=df, ax=axes[1],
            scatter_kws={'alpha': 0.4, 's': 30},
            line_kws={'color': 'red', 'linewidth': 2})
axes[1].set_title('Alcohol vs Quality (Regression)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Alcohol (%)', fontsize=12)
axes[1].set_ylabel('Quality Score', fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate correlation
corr = df['alcohol'].corr(df['quality'])
print(f'Correlation between Alcohol and Quality: {corr:.4f}')

In [ ]:
# Correlation matrix (Seaborn heatmap)
plt.figure(figsize=(14, 11))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8},
            vmin=-1, vmax=1)
plt.title('Correlation Matrix of All Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Display top correlations with quality
print('\nTop Correlations with Quality:')
print('='*80)
quality_corr = correlation_matrix['quality'].sort_values(ascending=False)
print('\nPositive Correlations:')
print(quality_corr[quality_corr > 0].drop('quality'))
print('\nNegative Correlations:')
print(quality_corr[quality_corr < 0])

## Complete Visualizations for Each Variable
Below are visualizations for all main features using Seaborn and Matplotlib.

## Patterns and Insights
- Higher alcohol content is generally associated with higher wine quality.
- Volatile acidity tends to decrease as quality increases, indicating less sourness in better wines.
- Wines with higher sulphates and lower density are more likely to have better quality scores.
- Most wines are rated between 5 and 7, showing a concentration in medium quality.
- There are strong correlations between some chemical features, such as free and total sulfur dioxide.
- These patterns can help producers optimize wine characteristics and can be used for predictive modeling.

In [ ]:
# Histogram for all numeric features
numeric_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(16, 12))
for i, col in enumerate(numeric_cols):
    plt.subplot(4, 3, i+1)
    sns.histplot(df[col], kde=True, color='skyblue')
    plt.title(f'Histogram of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for all numeric features
plt.figure(figsize=(16, 12))
for i, col in enumerate(numeric_cols):
    plt.subplot(4, 3, i+1)
    sns.boxplot(y=df[col], color='lightcoral')
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot for main features (Seaborn)
sns.pairplot(df, vars=['alcohol', 'quality', 'pH', 'citric acid', 'volatile acidity'], hue='quality', palette='coolwarm')
plt.suptitle('Pairplot of Selected Features', y=1.02)
plt.show()